## Quiz 05 - Parallel Computing, Reproducibility, and Containers

### Instructions

This quiz is based on the material covered in lectures 20 to 24. You may use any resources available to you, including the lecture notes and the internet.

All the data required for this quiz can be found in the `data` folder within this repository. If you need to recreate the datasets, you can do so by running the Python script included in the `script-data-generation` folder.

**Important:** Please start by completing Question 01 to set up the correct Python environment before proceeding with the other questions.

Answer the questions in the cells below.
When you are done, please submit your work as an `.html` file on Canvas, or share a link to a GitHub repository with your answers.

### **Question 01: Setting up the Python Environment**

Before proceeding with the rest of the quiz, it is important to set up a Python environment with specific package versions to ensure compatibility and reproducibility. This quiz requires **Python 3.12** and the following packages with exact versions:
- `dask=2026.3.0`
- `duckdb=1.5.1`
- `ipykernel=7.2.0`
- `joblib=1.5.3`
- `numpy=2.4.3`
- `pandas=3.0.2`
- `pyarrow=23.0.1`

You can use tools like `conda`, `pipenv`, or `uv` to manage your environment. If you use conda (recommended), please make sure you **create the environment and install all packages in the same command**. Also include `-c conda-forge` in your command. Make sure to change your current environment to the new environment after creation.

Write the terminal commands in the code cell below:

In [2]:
%%bash
conda create -n datasci350-quiz05 python=3.12 dask=2026.3.0 duckdb=1.5.1 ipykernel=7.2.0 joblib=1.5.3 numpy=2.4.3 pandas=3.0.2 pyarrow=23.0.1 -c conda-forge -y
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate datasci350-quiz05
python --version


2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done




==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda





## Package Plan ##

  environment location: /opt/anaconda3/envs/datasci350-quiz05

  added / updated specs:
    - dask=2026.3.0
    - duckdb=1.5.1
    - ipykernel=7.2.0
    - joblib=1.5.3
    - numpy=2.4.3
    - pandas=3.0.2
    - pyarrow=23.0.1
    - python=3.12


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |       7_kmp_llvm           8 KB  conda-forge
    _python_abi3_support-1.0   |       hd8ed1ab_2           8 KB  conda-forge
    appnope-0.1.4              |     pyhd8ed1ab_1          10 KB  conda-forge
    asttokens-3.0.1            |     pyhd8ed1ab_0          28 KB  conda-forge
    aws-c-auth-0.10.1          |       hcb83491_2         114 KB  conda-forge
    aws-c-cal-0.9.13           |       h6ee9776_1          44 KB  conda-forge
    aws-c-common-0.12.6        |       hc919400_0         219 KB  conda-forge
    aws-c-compression-0.3.2    |       h3e7

### Question 02: Understanding the `map` Function and Parallelism

The built-in Python `map()` function applies a function to each element sequentially. Using `joblib`, rewrite the following serial code to run in parallel using **all available cores**. Compare the results to verify correctness.

```python
import numpy as np

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)

# Serial version using map
serial_result = list(map(cube_root, numbers))
print("First 5 serial results:", serial_result[:5])
```

In [6]:
import numpy as np
from joblib import Parallel, delayed

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)



# serial 
serial_result = list(map(cube_root, numbers))

# parallel 
parallel_result = Parallel(n_jobs=-1)(
    delayed(cube_root)(x) for x in numbers
)

print("First 5 serial results:", serial_result[:5])
print("First 5 parallel results:", parallel_result[:5])
print("Same results or not:",np.allclose(serial_result, parallel_result))

First 5 serial results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]
First 5 parallel results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]
Same results or not: True


### Question 03: Measuring Parallel Speedup

Create a function called `simulate_computation` that generates 100,000 random numbers and calculates their variance. Using `%timeit`, measure and compare the execution time of:

1. Running the function **4 times sequentially** in a list comprehension (`[simulate_computation() for _ in range(4)]`)
2. Running the function **4 times in parallel** using `joblib` with 4 workers

Print and compare both timing results.

In [ ]:
# Please write your answer here.

def simulate_computation():
    x = np.random.random(100000)
    return np.var(x)

print("Sequential timing:")
%timeit [simulate_computation() for _ in range(4)]

print("Parallel timing:")
%timeit Parallel(n_jobs=4)(delayed(simulate_computation)() for _ in range(4))

# Parallel timing is slower because of overhead from managing small tasks

Sequential timing:
1.82 ms ± 25.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Parallel timing:
12.4 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Question 04: Dask Array with Custom Chunk Sizes

Create a Dask array of shape (5000, 2000) filled with random integers between 1 and 100. Use chunks of size (500, 500). Then:

1. Compute the sum of each row
2. Calculate the mean and standard deviation of the entire array
3. Print all three results

In [12]:
# Please write your answer here.
import dask.array as da

x = da.random.randint(1, 101, size=(5000, 2000), chunks=(500, 500))

row_sums = x.sum(axis=1).compute()
overall_mean = x.mean().compute()
overall_std = x.std().compute()

print("Row sums:", row_sums)
print("Mean:", overall_mean)
print("SD:", overall_std)

Row sums: [103155 100007 102713 ...  98363 100825  99568]
Mean: 50.4820319
SD: 28.857433107388164


### Question 05: Optimising Chunk Size

The chunk size significantly affects Dask performance. Create a Dask array with 100,000 random numbers and test three different chunk sizes: 1,000 (many small chunks), 10,000 (medium chunks), and 50,000 (few large chunks).

For each configuration, measure the time to compute `mean(sin(x) + cos(x))`. Which chunk size performed best? Explain why in a comment.

In [ ]:
# Please write your answer here.
x1 = da.random.random(100000, chunks=1000)
x2 = da.random.random(100000, chunks=10000)
x3 = da.random.random(100000, chunks=50000)

print("Chunks = 1000")
%timeit (da.sin(x1) + da.cos(x1)).mean().compute()

print("Chunks = 10000")
%timeit (da.sin(x2) + da.cos(x2)).mean().compute()

print("Chunks = 50000")
%timeit (da.sin(x3) + da.cos(x3)).mean().compute()

# Usually, medium chunks performs the best because very small chunks create too much overhead and very large chunks takes a lot to process
# But in this case, chunks=50000 performed best because the array is not very large and chunks=1000 was slowest because too many small chunks added more overhead

Chunks = 1000
42 ms ± 163 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Chunks = 10000
6.32 ms ± 131 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Chunks = 50000
3.26 ms ± 63.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Question 06: Reading Parquet Files with Column Selection

The `data` folder contains Parquet files for multiple countries. Using Dask, read **all Parquet files at once** (`data/*.parquet`), but load only the `year` and `population` columns.

Calculate the total world population for each year across all countries and display the results sorted by year.

In [15]:
# Please write your answer here.
import dask.dataframe as dd

df = dd.read_parquet("data/*.parquet", columns=["year", "population"])

result = (
    df.groupby("year")["population"]
      .sum()
      .reset_index()
      .sort_values("year")
      .compute()
)

print(result)

    year  population
0   1945   566994202
1   1946   596804909
2   1947   606569895
3   1948   637303888
4   1949   644613118
..   ...         ...
74  2019  1893887207
75  2020  1959057915
76  2021  1928046753
77  2022  1985837056
78  2023  1980706538

[79 rows x 2 columns]


### Question 07: DuckDB with Multiple Conditions

Load the `data.csv` file into a Pandas DataFrame. Using DuckDB, write a SQL query that:

1. Selects countries where `gdp_per_capita` was between 10000 and 50000
2. Filters for years between 2000 and 2020
3. Orders results by `gdp_per_capita` in descending order
4. Limits to the top 2 results

Execute the query and display the results.

In [16]:
# Please write your answer here.
import pandas as pd
import duckdb

df = pd.read_csv("data/data.csv")

result = duckdb.sql("""
    SELECT country, year, gdp_per_capita
    FROM df
    WHERE gdp_per_capita BETWEEN 10000 AND 50000
      AND year BETWEEN 2000 AND 2020
    ORDER BY gdp_per_capita DESC
    LIMIT 2
""").df()

print(result)

  country  year  gdp_per_capita
0     USA  2002    48942.492140
1     USA  2003    47607.365171


### Question 08: DuckDB with Aggregation

Using the same `data.csv` file, write a SQL query with DuckDB that calculates:

1. The average GDP per capita for each country
2. The minimum and maximum years in the dataset for each country

Group by country and display all results.

In [17]:
# Please write your answer here.

result2 = duckdb.sql("""
    SELECT
        country,
        AVG(gdp_per_capita) AS avg_gdp_per_capita,
        MIN(year) AS min_year,
        MAX(year) AS max_year
    FROM df
    GROUP BY country
    ORDER BY country
""").df()

print(result2)
    

  country  avg_gdp_per_capita  min_year  max_year
0  Brazil         5496.292031      1945      2023
1   India         1251.704443      1945      2023
2      UK        27496.851363      1945      2023
3     USA        40189.822290      1945      2023


### Question 09: Generating `requirements.txt` and `environment.yml` Files

Write the commands to:

1. Export your current environment's packages to a `requirements.txt` and an `environment.yml` file
2. Show how someone else would install these exact dependencies in these two cases

Explain each step with comments. It is not necessary to run the code.

In [ ]:
# Export installed Python packages from current environment to requirements.txt
# pip freeze > requirements.txt
# Export current conda environment to environment.yml
# conda env export --no-builds > environment.yml


# If someone wants to install from requirements.txt:
# 1. Create and activate a virtual environment
# python -m venv myenv
# source myenv/bin/activate
# 2. Install exact packages listed in requirements.txt
# pip install -r requirements.txt


# If someone wants to install from environment.yml:
# 1. Create the conda environment from the file
# conda env create -f environment.yml
# 2. Activate the created environment
# conda activate datasci350-quiz05

### Question 10: Troubleshooting a Broken Dockerfile

The following Dockerfile has several errors. Identify and fix 5 issues, then explain what was wrong with each line:

```dockerfile
# Broken Dockerfile - Fix the errors
from ubuntu

RUN apt install python3 python3-pip
RUN pip install numpy pandas

COPY . .
EXPOSE 8888
RUN ["python3", "app.py"]
```

Write the corrected Dockerfile and list each error with its fix.

```dockerfile
# Corrected Dockerfile 
FROM ubuntu:24.04 
# Dockerfile instructions should be uppercase. Tag added is better than using an unspecified latest image

RUN apt update && apt install -y python3 python3-pip
# apt should be updated first, and -y avoids an interactive install prompt

RUN pip3 install numpy pandas
# In Ubuntu, python3 is installed, so using pip3 is safer and matches the Python version being used

COPY . /app
# Copying into a defined working directory avoids saving into an unspecified default location

WORKDIR /app
EXPOSE 8888

CMD ["python3", "app.py"]
# RUN executes during image build, but CMD specifies the command to run when the container starts
```


### Question 11 - Writing a Dockerfile to Install Software on a Base Image

Create a Dockerfile that starts from an Ubuntu image and installs the following software:

- Git version 1:2.43.0-1ubuntu7.3
- SQLite version 3.45.1-1ubuntu2

Ensure that you specify the exact versions of the packages by checking their versions after installation. Include commands to clean up the package manager cache after installation to reduce the image size.

```dockerfile
FROM ubuntu:24.04

RUN apt update && \
    apt install -y \
    git=1:2.43.0-1ubuntu7.3 \
    sqlite3=3.45.1-1ubuntu2 && \
    git --version && \
    sqlite3 --version && \
    apt clean && \
    rm -rf /var/lib/apt/lists/*
```

### Question 12: Dockerfile for a Jupyter Data Science Environment

Create a Dockerfile starting from Ubuntu that:

1. Installs Python 3.12 and pip
2. Installs `jupyterlab`, `numpy`, `pandas`, `matplotlib`, and `scikit-learn` with specific versions of your choice
3. Sets the working directory to `/home/analyst/notebooks`
4. Exposes port 8888
5. Starts JupyterLab with `--no-browser` and `--ip=0.0.0.0`

Clean up apt cache to reduce image size.

```dockerfile
FROM ubuntu:24.04

RUN apt update && \
    apt install -y python3.12 python3-pip && \
    apt clean && \
    rm -rf /var/lib/apt/lists/*

RUN pip3 install \
    jupyterlab==4.2.5 \
    numpy==2.1.1 \
    pandas==2.2.3 \
    matplotlib==3.9.2 \
    scikit-learn==1.5.2

WORKDIR /home/analyst/notebooks
EXPOSE 8888
CMD ["jupyter-lab", "--no-browser", "--ip=0.0.0.0", "--port=8888", "--allow-root"]

```